# Working with Backend Graph Databases — User Guide

This guide is about *where the data actually lives* — which store `StarLayerGraph` talks to, and whether it speaks RDF 1.1 or native RDF 1.2 to get there. (RDF 1.2 *format* support — `turtle12`, `nt12`, `nq12`, `trig12`, `trix12`, `rdfxml12`, `jsonld12`, `longturtle12` — is a separate concern, covered in the [serialization formats guide](05a-serialization-formats.ipynb).)

- In-memory backend (default) and native RDF 1.2 backend modes
- Dual-mode operation: the same store can be driven in RDF 1.1 (encoding, rewritten queries) or native RDF 1.2 mode
- Any rdflib `Store` plugin works transparently under the default RDF 1.1 backend — not just the built-in in-memory store

The code snippets below for Oxigraph and Fuseki need a running instance and aren't executed *in this notebook* — but unlike earlier drafts of this guide, they're not merely "illustrative": the exact code paths shown (plain graph read/write including blank nodes, `apply_rules()`/`validate()` in any `meta_shacl` setting, in both backend modes) were confirmed live against both Oxigraph and Fuseki and are covered by this project's own integration test suite. See [`starlayergraph.md`](../../packages/graph/docs/starlayergraph.md) and [`compatibility.md`](../../packages/shacl/docs/compatibility.md) for the full detail behind that confirmation.

## How to run this notebook

1. `pip install "git+https://github.com/hidden-graph/starlayer.git"` (or install the three packages editable from a local checkout — see the root [README](../../README.md)).
2. Run cells from top to bottom — later sections reuse variables from earlier ones. Only the SQLAlchemy section actually executes here; the Oxigraph/Fuseki sections show code that needs a running instance (see each section for the `docker run` command).

In [1]:
from starlayergraph import StarLayerGraph, Namespace, TripleTerm

EX = Namespace("http://example.org/")
RDF = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")

## Backend compatibility at a glance

Confirmed live 2026-09-13, current status for `data_graph`/`shacl_graph` backed by a real remote SPARQL endpoint (Oxigraph, Fuseki) via `store=SPARQLUpdateStore(...)`, in either backend mode:

| Operation | In-memory (default) | Remote store, `rdf-1.1` mode | Remote store, `rdf-1.2` native mode |
| --- | --- | --- | --- |
| Plain graph read/write, including blank nodes | ✅ | ✅ | ✅ |
| `apply_rules()` / `validate()`, any `meta_shacl` setting | ✅ | ✅ | ✅ |

Blank-node read/write against a remote store required real fixes in `starlayergraph` itself (not just starshacl) — different mechanisms per mode, since rdflib's own `SPARQLUpdateStore` refuses outright to serialize *any* blank node into query/update text. See `starlayergraph.md`'s "Blank nodes against a remote store" section for the full mechanism in each mode, and `compatibility.md`'s "Backend Compatibility" section for what this means for `starshacl` specifically.

### Oxigraph — native RDF 1.2

Oxigraph 0.5.9+ speaks SPARQL 1.2 (triple-term syntax) directly over HTTP. Point `StarLayerGraph` at it with `backend='rdf-1.2'` and a `SPARQLUpdateStore`, and triple terms/direction-tagged literals go over the wire in their real syntax — no `tt:HASH` encoding, no query rewriting. Needs a running Oxigraph instance (`docker run -d -p 7878:7878 ghcr.io/oxigraph/oxigraph serve --location /data --bind 0.0.0.0:7878`); not executed in this notebook, but this exact pattern is exercised live in `packages/graph/tests/integration/test_oxigraph_backend.py` and `packages/shacl/tests/integration/test_native_backend_oxigraph.py`.

```python
from rdflib.plugins.stores.sparqlstore import SPARQLUpdateStore

store = SPARQLUpdateStore(
    query_endpoint="http://localhost:7878/query",
    update_endpoint="http://localhost:7878/update",
)
g = StarLayerGraph(store=store, identifier=EX.main, backend="rdf-1.2")
g.bind("ex", EX)
g.add_reification(EX.claim, TripleTerm(EX.bob, EX.knows, EX.carol))

rows = g.query("""
    PREFIX ex: <http://example.org/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    SELECT ?stmt WHERE { ?stmt rdf:reifies <<( ex:bob ex:knows ex:carol )>> }
""")
for row in rows:
    print(g.qname(row.stmt))
```

### Fuseki — dual-mode: RDF 1.1 and RDF 1.2 against the same store

Fuseki 5.5+ also speaks native RDF 1.2. The dual-mode story: the *same* store connection works in either backend mode — omit `backend=` (defaults to `'rdf-1.1'`) to have StarLayer encode triple terms and rewrite SPARQL 1.2 syntax down to plain SPARQL 1.1 before sending it, for compatibility with any SPARQL 1.1-only endpoint; pass `backend='rdf-1.2'` to send native RDF 1.2 syntax directly once you know the endpoint supports it. Needs a running Fuseki instance (`docker run -d -p 3030:3030 atomgraph/fuseki:latest --update --mem --ping /starlayergraph`, Fuseki 5.5+ required for the RDF 1.2 mode); not executed here, but exercised live in `packages/graph/tests/integration/test_fuseki_backend.py` and `packages/shacl/tests/integration/test_native_backend_fuseki.py`.

One Fuseki-specific detail worth knowing if you compare raw blank node labels across separate queries yourself: Fuseki resets its blank-node label counter *per query* (two different stored nodes can each come back labeled `"b0"` in separate requests), unlike Oxigraph's stable, content-derived labels. `starlayergraph` already accounts for this — see `starlayergraph.md`'s "Blank nodes against a remote store" section — but it's the reason naive client-side label-matching across queries is unsound on Fuseki specifically.

```python
from rdflib.plugins.stores.sparqlstore import SPARQLUpdateStore

def make_store():
    return SPARQLUpdateStore(
        query_endpoint="http://localhost:3030/starlayergraph/query",
        update_endpoint="http://localhost:3030/starlayergraph/update",
        auth=("admin", "admin"),
    )

# RDF 1.1 mode: triple terms encoded, SPARQL 1.2 rewritten to 1.1 before sending
g11 = StarLayerGraph(store=make_store(), identifier=EX.main)   # backend='rdf-1.1' is the default
g11.bind("ex", EX)
g11.add_reification(EX.claim, TripleTerm(EX.bob, EX.knows, EX.carol))

# RDF 1.2 mode: native triple-term syntax sent directly, no rewriting
g12 = StarLayerGraph(store=make_store(), identifier=EX.main, backend="rdf-1.2")
g12.bind("ex", EX)
rows = g12.query("""
    PREFIX ex: <http://example.org/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    SELECT ?stmt WHERE { ?stmt rdf:reifies <<( ex:bob ex:knows ex:carol )>> }
""")
for row in rows:
    print(g12.qname(row.stmt))
```

### SQLAlchemy — RDF 1.1, real SQL-backed persistence

Unlike Oxigraph/Fuseki above, this one *is* executed in this notebook: `StarLayerGraph` is an ordinary `rdflib.Graph` subclass, so any rdflib `Store` plugin works transparently under the default RDF 1.1 (encoding) backend — including a real SQL database via `rdflib-sqlalchemy`. Needs the `sqlalchemy` extra (`pip install rdflib-sqlalchemy`); no native RDF 1.2 mode exists for this store (`rdflib-sqlalchemy` has no SPARQL 1.2 support of its own), so this is RDF 1.1-only.

In [2]:
!pip install -q rdflib-sqlalchemy

import tempfile
import rdflib_sqlalchemy
rdflib_sqlalchemy.registerplugins()

db_path = tempfile.mktemp(suffix=".sqlite")
uri = f"sqlite:///{db_path}"

writer = StarLayerGraph(store="SQLAlchemy", identifier=EX.main)
writer.open(uri, create=True)
writer.bind("ex", EX)
writer.add_reification(EX.claim, TripleTerm(EX.bob, EX.knows, EX.carol))
writer.commit()
writer.close()

# fresh graph object, same database file - proves the data actually persisted
reader = StarLayerGraph(store="SQLAlchemy", identifier=EX.main)
reader.open(uri, create=False)
reader.bind("ex", EX)
print(reader.qname_term(next(reader.triple_terms(subject=EX.bob))))
reader.close()

<<( ex:bob ex:knows ex:carol )>>


/Users/johnclements/Documents/GitHub/hidden-graph/starlayer/.venv/lib/python3.14/site-packages/rdflib_sqlalchemy/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution


## Further work

- **Non-Oxigraph/Fuseki SPARQL 1.1 endpoints and SQL backends beyond SQLAlchemy.** This project's testing is specifically against Oxigraph and Fuseki; other SPARQL 1.1-conformant stores should work under the default `rdf-1.1` backend (it only relies on standard SPARQL 1.1 behavior), but haven't been verified directly.
- **`meta_shacl=True` against a remote store** works by snapshotting the shapes graph in-memory first, rather than running meta-shacl against the remote store directly — see `compatibility.md`'s "Backend Compatibility" section for why that's the right fix rather than a workaround.